# Custom Dataset Integration Tutorial

This notebook demonstrates how to integrate your own custom datasets with the SMOTE for Images pipeline. We'll cover data loading, preprocessing, and pipeline configuration for different types of image datasets.

## Supported Dataset Formats

1. **Folder Structure**: Images organized in class folders
2. **CSV Annotations**: Image paths with labels in CSV format
3. **PyTorch Datasets**: Custom PyTorch dataset classes
4. **NumPy Arrays**: Pre-loaded image arrays with labels

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import numpy as np
import pandas as pd
from PIL import Image
from pathlib import Path
import matplotlib.pyplot as plt
from collections import Counter
import os

# Import SMOTE components
from smote_image_synthesis.pipeline import SynthesisPipeline
from smote_image_synthesis.data.preprocessor import ImagePreprocessor
from smote_image_synthesis.encoders.resnet_encoder import ResNetEncoder
from smote_image_synthesis.decoders.autoencoder_decoder import AutoencoderDecoder
from smote_image_synthesis.smote.constrained_smote import ConstrainedSMOTE
from smote_image_synthesis.quality.assessor import QualityAssessor

## 1. Dataset Format Examples

Let's create example datasets in different formats to demonstrate integration.

In [ ]:
# Create example dataset directory structure
def create_example_datasets():
    """
    Create example datasets in different formats for demonstration.
    """
    base_dir = Path("./example_datasets")
    base_dir.mkdir(exist_ok=True)
    
    # 1. Folder structure dataset
    folder_dataset_dir = base_dir / "folder_structure"
    for class_name in ['cats', 'dogs', 'birds']:
        (folder_dataset_dir / class_name).mkdir(parents=True, exist_ok=True)
    
    # 2. CSV dataset directory
    csv_dataset_dir = base_dir / "csv_format"
    csv_dataset_dir.mkdir(exist_ok=True)
    
    # Generate sample images and metadata
    image_data = []
    
    for class_id, class_name in enumerate(['cats', 'dogs', 'birds']):
        # Create imbalanced distribution
        n_samples = [100, 30, 15][class_id]  # Imbalanced classes
        
        for i in range(n_samples):
            # Generate synthetic image
            img_array = np.random.randint(0, 255, (64, 64, 3), dtype=np.uint8)
            
            # Add class-specific patterns
            if class_id == 0:  # cats - add some structure
                img_array[20:40, 20:40] = [255, 200, 150]  # Orange patch
            elif class_id == 1:  # dogs
                img_array[10:50, 10:50] = [150, 100, 50]   # Brown patch
            else:  # birds
                img_array[15:45, 15:45] = [100, 150, 255]  # Blue patch
            
            img = Image.fromarray(img_array)
            
            # Save for folder structure dataset
            folder_path = folder_dataset_dir / class_name / f"{i:04d}.jpg"
            img.save(folder_path)
            
            # Save for CSV dataset
            csv_path = csv_dataset_dir / f"{class_name}_{i:04d}.jpg"
            img.save(csv_path)
            
            # Add to CSV metadata
            image_data.append({
                'image_path': str(csv_path),
                'class_name': class_name,
                'class_id': class_id,
                'split': 'train' if i < n_samples * 0.8 else 'val'
            })
    
    # Create CSV annotation file
    df = pd.DataFrame(image_data)
    df.to_csv(base_dir / "annotations.csv", index=False)
    
    print(f"Created example datasets in {base_dir}")
    print(f"Class distribution: {Counter(df['class_name'])}")
    
    return base_dir

# Create example datasets
dataset_dir = create_example_datasets()

## 2. Method 1: Folder Structure Dataset

Load dataset from folder structure where each class has its own directory.

In [ ]:
class FolderDataset(Dataset):
    """
    Dataset class for folder-structured image datasets.
    """
    
    def __init__(self, root_dir, transform=None, target_size=(64, 64)):
        self.root_dir = Path(root_dir)
        self.transform = transform
        self.target_size = target_size
        
        # Get class names from folder names
        self.classes = sorted([d.name for d in self.root_dir.iterdir() if d.is_dir()])
        self.class_to_idx = {cls: idx for idx, cls in enumerate(self.classes)}
        
        # Collect all image paths and labels
        self.samples = []
        for class_name in self.classes:
            class_dir = self.root_dir / class_name
            class_idx = self.class_to_idx[class_name]
            
            for img_path in class_dir.glob('*.jpg'):
                self.samples.append((str(img_path), class_idx))
        
        print(f"Found {len(self.samples)} images in {len(self.classes)} classes")
        
        # Default transform if none provided
        if self.transform is None:
            self.transform = transforms.Compose([
                transforms.Resize(self.target_size),
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                                   std=[0.229, 0.224, 0.225])
            ])
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        
        # Load image
        image = Image.open(img_path).convert('RGB')
        
        if self.transform:
            image = self.transform(image)
        
        return image, label
    
    def get_class_distribution(self):
        """Get class distribution for analysis."""
        labels = [label for _, label in self.samples]
        return Counter(labels)

# Load folder dataset
folder_dataset = FolderDataset(dataset_dir / "folder_structure")
print(f"Classes: {folder_dataset.classes}")
print(f"Distribution: {folder_dataset.get_class_distribution()}")

# Create data loader
folder_loader = DataLoader(folder_dataset, batch_size=32, shuffle=True)

# Visualize some samples
def visualize_batch(dataset, n_samples=8):
    fig, axes = plt.subplots(2, 4, figsize=(12, 6))
    axes = axes.flatten()
    
    for i in range(n_samples):
        image, label = dataset[i]
        
        # Denormalize for visualization
        mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
        std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
        image = image * std + mean
        image = torch.clamp(image, 0, 1)
        
        axes[i].imshow(image.permute(1, 2, 0))
        axes[i].set_title(f'Class: {folder_dataset.classes[label]}')
        axes[i].axis('off')
    
    plt.tight_layout()
    plt.show()

visualize_batch(folder_dataset)

## 3. Method 2: CSV Annotations Dataset

Load dataset using CSV file with image paths and labels.

In [ ]:
class CSVDataset(Dataset):
    """
    Dataset class for CSV-annotated image datasets.
    """
    
    def __init__(self, csv_file, transform=None, split='train', target_size=(64, 64)):
        self.df = pd.read_csv(csv_file)
        
        # Filter by split if specified
        if split:
            self.df = self.df[self.df['split'] == split]
        
        self.transform = transform
        self.target_size = target_size
        
        # Get unique classes
        self.classes = sorted(self.df['class_name'].unique())
        self.class_to_idx = {cls: idx for idx, cls in enumerate(self.classes)}
        
        print(f"Loaded {len(self.df)} images for split '{split}'")
        
        # Default transform
        if self.transform is None:
            self.transform = transforms.Compose([
                transforms.Resize(self.target_size),
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                                   std=[0.229, 0.224, 0.225])
            ])
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        # Load image
        image = Image.open(row['image_path']).convert('RGB')
        label = self.class_to_idx[row['class_name']]
        
        if self.transform:
            image = self.transform(image)
        
        return image, label
    
    def get_class_distribution(self):
        """Get class distribution for analysis."""
        return Counter(self.df['class_name'])

# Load CSV dataset
train_csv_dataset = CSVDataset(dataset_dir / "annotations.csv", split='train')
val_csv_dataset = CSVDataset(dataset_dir / "annotations.csv", split='val')

print(f"Train distribution: {train_csv_dataset.get_class_distribution()}")
print(f"Val distribution: {val_csv_dataset.get_class_distribution()}")

# Create data loaders
train_csv_loader = DataLoader(train_csv_dataset, batch_size=32, shuffle=True)
val_csv_loader = DataLoader(val_csv_dataset, batch_size=32, shuffle=False)

## 4. Method 3: Custom PyTorch Dataset

Create a custom dataset class with advanced preprocessing and augmentation.

In [ ]:
class AdvancedImageDataset(Dataset):
    """
    Advanced dataset class with custom preprocessing and augmentation.
    """
    
    def __init__(self, images, labels, transform=None, augment=False, target_size=(64, 64)):
        self.images = images
        self.labels = labels
        self.target_size = target_size
        self.augment = augment
        
        # Base transform
        base_transforms = [
            transforms.Resize(target_size),
            transforms.ToTensor()
        ]
        
        # Add augmentation if requested
        if augment:
            augment_transforms = [
                transforms.RandomHorizontalFlip(p=0.5),
                transforms.RandomRotation(degrees=15),
                transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
                transforms.RandomAffine(degrees=0, translate=(0.1, 0.1))
            ]
            base_transforms = augment_transforms + base_transforms
        
        # Add normalization
        base_transforms.append(
            transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                               std=[0.229, 0.224, 0.225])
        )
        
        self.transform = transform or transforms.Compose(base_transforms)
        
        # Get unique classes
        self.classes = sorted(list(set(labels)))
        self.class_to_idx = {cls: idx for idx, cls in enumerate(self.classes)}
        
        print(f"Created dataset with {len(self.images)} images")
        print(f"Augmentation: {augment}")
    
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        image = self.images[idx]
        label = self.labels[idx]
        
        # Convert to PIL if numpy array
        if isinstance(image, np.ndarray):
            image = Image.fromarray(image)
        
        if self.transform:
            image = self.transform(image)
        
        return image, label
    
    def get_class_distribution(self):
        return Counter(self.labels)
    
    def balance_classes(self, method='oversample'):
        """
        Balance classes using simple oversampling or undersampling.
        """
        class_counts = self.get_class_distribution()
        
        if method == 'oversample':
            max_count = max(class_counts.values())
            target_count = max_count
        else:  # undersample
            min_count = min(class_counts.values())
            target_count = min_count
        
        balanced_images = []
        balanced_labels = []
        
        for class_label in self.classes:
            # Get indices for this class
            class_indices = [i for i, label in enumerate(self.labels) if label == class_label]
            
            # Sample to target count
            if len(class_indices) < target_count:
                # Oversample
                sampled_indices = np.random.choice(class_indices, target_count, replace=True)
            else:
                # Undersample
                sampled_indices = np.random.choice(class_indices, target_count, replace=False)
            
            for idx in sampled_indices:
                balanced_images.append(self.images[idx])
                balanced_labels.append(self.labels[idx])
        
        return AdvancedImageDataset(balanced_images, balanced_labels, 
                                  transform=self.transform, augment=self.augment)

# Convert folder dataset to arrays for demonstration
def dataset_to_arrays(dataset):
    images = []
    labels = []
    
    for i in range(len(dataset)):
        # Get raw image without transform
        img_path, label = dataset.samples[i]
        image = Image.open(img_path).convert('RGB')
        images.append(image)
        labels.append(label)
    
    return images, labels

# Create advanced dataset
images, labels = dataset_to_arrays(folder_dataset)
advanced_dataset = AdvancedImageDataset(images, labels, augment=True)

print(f"Original distribution: {advanced_dataset.get_class_distribution()}")

# Balance the dataset
balanced_dataset = advanced_dataset.balance_classes(method='oversample')
print(f"Balanced distribution: {balanced_dataset.get_class_distribution()}")

## 5. Integration with SMOTE Pipeline

Now let's integrate our custom dataset with the SMOTE pipeline.

In [ ]:
def prepare_data_for_pipeline(dataset, batch_size=32):
    """
    Convert dataset to tensors for pipeline usage.
    """
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False)
    
    all_images = []
    all_labels = []
    
    for images, labels in dataloader:
        all_images.append(images)
        all_labels.append(labels)
    
    images_tensor = torch.cat(all_images, dim=0)
    labels_array = torch.cat(all_labels, dim=0).numpy()
    
    return images_tensor, labels_array

# Prepare data
train_images, train_labels = prepare_data_for_pipeline(train_csv_dataset)
val_images, val_labels = prepare_data_for_pipeline(val_csv_dataset)

print(f"Train data: {train_images.shape}, {len(train_labels)} labels")
print(f"Val data: {val_images.shape}, {len(val_labels)} labels")
print(f"Train class distribution: {Counter(train_labels)}")

# Configure pipeline components
encoder = ResNetEncoder(
    architecture='resnet18',
    embedding_dim=256,
    pretrained=True,
    freeze_backbone=True
)

decoder = AutoencoderDecoder(
    embedding_dim=256,
    image_shape=(3, 64, 64),
    hidden_dims=[512, 256, 128]
)

smote = ConstrainedSMOTE(
    k_neighbors=5,
    sampling_strategy='auto',
    use_clustering=True,
    clustering_method='kmeans'
)

quality_assessor = QualityAssessor(
    metrics=['mse', 'ssim', 'psnr'],
    compute_diversity=True
)

# Create pipeline
pipeline = SynthesisPipeline(
    encoder=encoder,
    decoder=decoder,
    smote=smote,
    quality_assessor=quality_assessor
)

print("Pipeline created successfully!")

## 6. Train Pipeline on Custom Dataset

Train the pipeline using our custom dataset.

In [ ]:
# Train the pipeline
print("Training pipeline on custom dataset...")

pipeline.fit(
    images=train_images,
    labels=train_labels,
    train_decoder=True,
    decoder_epochs=15,
    validation_data=(val_images, val_labels)
)

print("Pipeline training completed!")

# Generate synthetic images
n_synthetic = 30
synthetic_images, synthetic_labels = pipeline.generate_synthetic_images(
    n_samples=n_synthetic
)

print(f"Generated {len(synthetic_images)} synthetic images")
print(f"Synthetic class distribution: {Counter(synthetic_labels)}")

## 7. Evaluate Results

Evaluate the quality of synthetic images and analyze class balance improvement.

In [ ]:
# Quality evaluation
quality_results = pipeline.evaluate_quality(
    synthetic_images=synthetic_images,
    real_images=val_images[:len(synthetic_images)]
)

print("Quality Metrics:")
for metric, value in quality_results['metrics'].items():
    print(f"  {metric.upper()}: {value:.4f}")

if 'diversity' in quality_results:
    print("\nDiversity Metrics:")
    for metric, value in quality_results['diversity'].items():
        print(f"  {metric}: {value:.4f}")

# Visualize results
def plot_class_balance_improvement(original_labels, synthetic_labels, class_names):
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(15, 4))
    
    # Original distribution
    original_counts = Counter(original_labels)
    classes = list(range(len(class_names)))
    original_values = [original_counts.get(i, 0) for i in classes]
    
    ax1.bar(classes, original_values, alpha=0.7, color='blue')
    ax1.set_title('Original Distribution')
    ax1.set_xlabel('Class')
    ax1.set_ylabel('Count')
    ax1.set_xticks(classes)
    ax1.set_xticklabels(class_names, rotation=45)
    
    # Synthetic distribution
    synthetic_counts = Counter(synthetic_labels)
    synthetic_values = [synthetic_counts.get(i, 0) for i in classes]
    
    ax2.bar(classes, synthetic_values, alpha=0.7, color='red')
    ax2.set_title('Synthetic Distribution')
    ax2.set_xlabel('Class')
    ax2.set_ylabel('Count')
    ax2.set_xticks(classes)
    ax2.set_xticklabels(class_names, rotation=45)
    
    # Combined distribution
    combined_labels = np.concatenate([original_labels, synthetic_labels])
    combined_counts = Counter(combined_labels)
    combined_values = [combined_counts.get(i, 0) for i in classes]
    
    ax3.bar(classes, combined_values, alpha=0.7, color='green')
    ax3.set_title('Combined Distribution')
    ax3.set_xlabel('Class')
    ax3.set_ylabel('Count')
    ax3.set_xticks(classes)
    ax3.set_xticklabels(class_names, rotation=45)
    
    plt.tight_layout()
    plt.show()
    
    # Print statistics
    print(f"Original: {dict(original_counts)}")
    print(f"Synthetic: {dict(synthetic_counts)}")
    print(f"Combined: {dict(combined_counts)}")

# Plot class balance improvement
class_names = train_csv_dataset.classes
plot_class_balance_improvement(train_labels, synthetic_labels, class_names)

## 8. Visual Comparison

Compare original and synthetic images side by side.

In [ ]:
def plot_original_vs_synthetic(original_images, synthetic_images, 
                              original_labels, synthetic_labels, 
                              class_names, n_samples=8):
    fig, axes = plt.subplots(2, n_samples, figsize=(16, 6))
    
    # Denormalization function
    def denormalize(tensor):
        mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
        std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
        return torch.clamp(tensor * std + mean, 0, 1)
    
    # Plot original images
    for i in range(min(n_samples, len(original_images))):
        img = denormalize(original_images[i])
        axes[0, i].imshow(img.permute(1, 2, 0))
        axes[0, i].set_title(f'Original\n{class_names[original_labels[i]]}')
        axes[0, i].axis('off')
    
    # Plot synthetic images
    for i in range(min(n_samples, len(synthetic_images))):
        img = denormalize(synthetic_images[i])
        axes[1, i].imshow(img.permute(1, 2, 0))
        axes[1, i].set_title(f'Synthetic\n{class_names[synthetic_labels[i]]}')
        axes[1, i].axis('off')
    
    plt.tight_layout()
    plt.show()

# Plot comparison
plot_original_vs_synthetic(
    val_images, synthetic_images,
    val_labels, synthetic_labels,
    class_names
)

## 9. Save and Export Results

Save the trained pipeline and export synthetic images.

In [ ]:
# Create output directory
output_dir = Path("./custom_dataset_results")
output_dir.mkdir(exist_ok=True)

# Save pipeline
pipeline.save_pipeline(output_dir / "trained_pipeline")
print(f"Pipeline saved to {output_dir / 'trained_pipeline'}")

# Save synthetic images
synthetic_dir = output_dir / "synthetic_images"
synthetic_dir.mkdir(exist_ok=True)

def save_synthetic_images(images, labels, class_names, output_dir):
    """
    Save synthetic images organized by class.
    """
    # Denormalization
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    
    for i, (image, label) in enumerate(zip(images, labels)):
        # Denormalize
        image = torch.clamp(image * std + mean, 0, 1)
        
        # Convert to PIL
        image_pil = transforms.ToPILImage()(image)
        
        # Create class directory
        class_dir = output_dir / class_names[label]
        class_dir.mkdir(exist_ok=True)
        
        # Save image
        image_path = class_dir / f"synthetic_{i:04d}.jpg"
        image_pil.save(image_path)
    
    print(f"Saved {len(images)} synthetic images to {output_dir}")

save_synthetic_images(synthetic_images, synthetic_labels, class_names, synthetic_dir)

# Save quality report
report_path = output_dir / "quality_report.txt"
with open(report_path, 'w') as f:
    f.write("Custom Dataset SMOTE Results\n")
    f.write("=" * 30 + "\n\n")
    
    f.write("Dataset Information:\n")
    f.write(f"Classes: {class_names}\n")
    f.write(f"Original distribution: {Counter(train_labels)}\n")
    f.write(f"Synthetic distribution: {Counter(synthetic_labels)}\n\n")
    
    f.write("Quality Metrics:\n")
    for metric, value in quality_results['metrics'].items():
        f.write(f"{metric.upper()}: {value:.4f}\n")
    
    if 'diversity' in quality_results:
        f.write("\nDiversity Metrics:\n")
        for metric, value in quality_results['diversity'].items():
            f.write(f"{metric}: {value:.4f}\n")

print(f"Quality report saved to {report_path}")

## 10. Best Practices and Tips

Here are some best practices for integrating custom datasets:

In [ ]:
def print_best_practices():
    print("=== CUSTOM DATASET INTEGRATION BEST PRACTICES ===")
    print()
    
    print("📁 DATA ORGANIZATION:")
    print("• Use consistent naming conventions")
    print("• Organize images in class folders or use CSV annotations")
    print("• Ensure balanced validation sets for proper evaluation")
    print("• Keep original and processed data separate")
    print()
    
    print("🖼️ IMAGE PREPROCESSING:")
    print("• Resize images to consistent dimensions")
    print("• Normalize using ImageNet statistics for pretrained encoders")
    print("• Apply appropriate data augmentation for training")
    print("• Handle different image formats (RGB, grayscale, etc.)")
    print()
    
    print("⚖️ CLASS IMBALANCE:")
    print("• Analyze class distribution before applying SMOTE")
    print("• Consider domain-specific sampling strategies")
    print("• Validate that synthetic samples are semantically meaningful")
    print("• Monitor quality metrics for minority classes")
    print()
    
    print("🔧 PIPELINE CONFIGURATION:")
    print("• Start with smaller models for prototyping")
    print("• Adjust embedding dimensions based on dataset complexity")
    print("• Use appropriate decoder architecture for your domain")
    print("• Tune SMOTE parameters based on embedding space analysis")
    print()
    
    print("📊 EVALUATION:")
    print("• Use multiple quality metrics for comprehensive assessment")
    print("• Evaluate both pixel-level and perceptual quality")
    print("• Assess diversity within and across classes")
    print("• Validate synthetic images with domain experts")
    print()
    
    print("💾 REPRODUCIBILITY:")
    print("• Save pipeline configurations and random seeds")
    print("• Document preprocessing steps and parameters")
    print("• Version control datasets and model checkpoints")
    print("• Create detailed experiment logs")

print_best_practices()

## Conclusion

This notebook demonstrated how to integrate custom datasets with the SMOTE for Images pipeline:

### Key Integration Methods

1. **Folder Structure**: Simple organization with class directories
2. **CSV Annotations**: Flexible metadata-driven approach
3. **Custom PyTorch Datasets**: Advanced preprocessing and augmentation
4. **Direct Array Integration**: For pre-processed data

### Integration Steps

1. **Data Loading**: Choose appropriate dataset class for your format
2. **Preprocessing**: Apply consistent transformations and normalization
3. **Pipeline Setup**: Configure components based on dataset characteristics
4. **Training**: Fit pipeline with proper validation
5. **Generation**: Create synthetic images for minority classes
6. **Evaluation**: Assess quality and class balance improvement

### Next Steps

- Adapt the examples to your specific dataset format
- Experiment with different preprocessing strategies
- Fine-tune pipeline parameters for your domain
- Implement domain-specific quality metrics
- Scale to larger datasets with batch processing

The pipeline is designed to be flexible and can handle various image datasets with minimal modifications to the core components.